In [ ]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState,START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

load_dotenv()

model =ChatGroq(
 model="openai/gpt-oss-120b",
 api_key=os.environ["GROQ_API_KEY"],
 streaming=True
)

def chatbot(state:MessagesState):
 response =model.invoke(state["messages"])
 return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot",chatbot)
builder.add_edge(START,"chatbot")
builder.add_edge("chatbot", END)

conn= sqlite3.connect(
"chat_memory.sqlite",
 check_same_thread=False
)

checkpointer = SqliteSaver(conn)
app = builder.compile(checkpointer=checkpointer)
config ={
    "configurable":{"thread_id":"chat-1"}
    }


while True:
    user_input = input("\nYou:")
    if user_input.lower() in ["exit","quit"]:
        print("Chat ended!")
    break

#It will not waitining in the buffer
    print("AI: ",end="", flush=True)

for message, metadata in app.stream(
 {"messages": [HumanMessage(content=user_input)]},
 config=config,
 stream_mode="messages"
 ):
    print(message.content,end="", flush=True)
    print()


